## Task-1

In [25]:
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split


In [26]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [29]:
print("Loading dataset...")
df = pd.read_csv('movies.csv')
allowed_cols = ['overview', 'tagline', 'keywords', 'genres', 'vote_average']



Loading dataset...


In [30]:
df =  df[allowed_cols].copy()

df['overview'] = df['overview'].fillna("")
df['tagline'] = df['tagline'].fillna("")
df['keywords'] = df['keywords'].fillna("")


In [ ]:
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    tokens = text.split()
    
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(lemmatized_tokens)


In [ ]:
print("Cleaning text columns... (this might take a minute depending on the dataset size)")
df['overview_clean'] = df['overview'].apply(clean_text)
df['tagline_clean'] = df['tagline'].apply(clean_text)
df['keywords_clean'] = df['keywords'].apply(clean_text)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)

val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
print(f"Data Splits Created Successfully:")
print(f"Train set: {len(train_df)} rows")
print(f"Validation set: {len(val_df)} rows")
print(f"Test set: {len(test_df)} rows")

Cleaning text columns... (this might take a minute depending on the dataset size)
Data Splits Created Successfully:
Train set: 3362 rows
Validation set: 720 rows
Test set: 721 rows


## Task 2

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

glove_file_path = "wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"
DIMENSION = 100

In [ ]:
def load_glove_model(filepath, embed_dim):
    print("Loading GloVe Model... (This takes a moment)")
    glove_model = {}
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            
            try:
                split_line = line.split()
                
                
                if len(split_line) < embed_dim + 1:
                    continue
                
                word = "".join(split_line[:-embed_dim])
                
                # Convert to float
                embedding = np.array([float(val) for val in split_line[-embed_dim:]], dtype=np.float32)
                glove_model[word] = embedding
                
            except ValueError:
                
                continue
                
    print(f"Done. {len(glove_model)} words successfully loaded!")
    return glove_model

In [41]:
glove_dict = load_glove_model("wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt", 100)

Loading GloVe Model... (This takes a moment)
Done. 1290970 words successfully loaded!


In [ ]:
def create_document_embeddings(train_text, val_text, test_text, glove_model, embed_dim):
    tfidf = TfidfVectorizer()
    
    
    X_train_tfidf = tfidf.fit_transform(train_text)
    X_val_tfidf = tfidf.transform(val_text)
    X_test_tfidf = tfidf.transform(test_text)
    
    vocab = tfidf.vocabulary_ 
    
    
    dataset_words = set(vocab.keys())
    glove_words = set(glove_model.keys())
    
    overlap = dataset_words.intersection(glove_words)
    coverage_percentage = (len(overlap) / len(dataset_words)) * 100 if len(dataset_words) > 0 else 0
    
    print(f"\n--- Embedding Coverage Report ---")
    print(f"Dataset Unique Words (from Train set): {len(dataset_words)}")
    print(f"Words Found in GloVe: {len(overlap)}")
    print(f"Coverage: {coverage_percentage:.2f}%")
    
    
    vocab_size = len(vocab)
    embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
    
    for word, idx in vocab.items():
        if word in glove_model:
            embedding_matrix[idx] = glove_model[word]
            
    print("\nCalculating TF-IDF Weighted Averages...")
    
    train_embeddings = X_train_tfidf.dot(embedding_matrix)
    val_embeddings = X_val_tfidf.dot(embedding_matrix)
    test_embeddings = X_test_tfidf.dot(embedding_matrix)
    
    train_weights_sum = X_train_tfidf.sum(axis=1) + 1e-9
    train_embeddings = train_embeddings / train_weights_sum
    
    val_weights_sum = X_val_tfidf.sum(axis=1) + 1e-9
    val_embeddings = val_embeddings / val_weights_sum
    
    test_weights_sum = X_test_tfidf.sum(axis=1) + 1e-9
    test_embeddings = test_embeddings / test_weights_sum
    
    return train_embeddings, val_embeddings, test_embeddings

In [44]:
train_overview_emb, val_overview_emb, test_overview_emb = create_document_embeddings(
    train_text=train_df['overview_clean'],
    val_text=val_df['overview_clean'],
    test_text=test_df['overview_clean'],
    glove_model=glove_dict,
    embed_dim=DIMENSION
)
print(f"\nFinal Overview Output Vector Shape (Movies, Dimension): {train_overview_emb.shape}")


--- Embedding Coverage Report ---
Dataset Unique Words (from Train set): 15763
Words Found in GloVe: 15596
Coverage: 98.94%

Calculating TF-IDF Weighted Averages...

Final Overview Output Vector Shape (Movies, Dimension): (3362, 100)


## Taask-3

In [45]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error

In [ ]:

X_train_over_tensor = torch.tensor(train_overview_emb, dtype=torch.float32)
y_train_tensor = torch.tensor(train_df['vote_average'].values, dtype=torch.float32).unsqueeze(1) # Make sure it's a column of numbers!
X_val_over_tensor = torch.tensor(val_overview_emb, dtype=torch.float32)
y_val_tensor = torch.tensor(val_df['vote_average'].values, dtype=torch.float32).unsqueeze(1)
X_test_over_tensor = torch.tensor(test_overview_emb, dtype=torch.float32)
y_test_tensor = torch.tensor(test_df['vote_average'].values, dtype=torch.float32).unsqueeze(1)

train_loader_over = DataLoader(TensorDataset(X_train_over_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader_over = DataLoader(TensorDataset(X_val_over_tensor, y_val_tensor), batch_size=64)
test_loader_over = DataLoader(TensorDataset(X_test_over_tensor, y_test_tensor), batch_size=64)


In [49]:
class RatingRegressor(nn.Module):
    def __init__(self, input_dim):
        super(RatingRegressor, self).__init__()
        # A simple model: Input (100) -> Hidden Layer (64 Neurons) -> Output (1 Rating)
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        # Dropout randomly turns off 30% of neurons to prevent overfitting (memorizing the dataset blindly)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 1)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [55]:
def train_and_evaluate_regressor(train_loader, val_loader, test_loader, input_name):
    # Initialize our model, the MSE logic, and the Adam Optimizer
    model = RatingRegressor(input_dim=DIMENSION)
    criterion = nn.MSELoss() 
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    epochs = 10 # Train by reading the whole dataset 10 times
    print(f"\n--- Training Regressor on column: '{input_name}' ---")
    
    for epoch in range(epochs):
        model.train() # Tell PyTorch we are actively learning
        total_train_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()      # Wipe old memory
            predictions = model(batch_X) # Make a guess
            loss = criterion(predictions, batch_y) # Calculate how wrong it was
            loss.backward()            # Calculate the direction of the error mathematically 
            optimizer.step()           # Adjust the neurons to be slightly smarter
            total_train_loss += loss.item()
            
        # Validation Phase (Testing without adjusting weights)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                preds = model(batch_X)
                val_loss += criterion(preds, batch_y).item()
                
        # Optional: uncomment to watch it get smarter live
        # print(f"Epoch {epoch+1}/{epochs} - Train MSE: {total_train_loss/len(train_loader):.3f} | Val MSE: {val_loss/len(val_loader):.3f}")
    print("Finished Training.")
    model.eval()
    test_preds = []
    test_actuals = []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            preds = model(batch_X)
            test_preds.extend(preds.squeeze().tolist())
            test_actuals.extend(batch_y.squeeze().tolist())
            
    mse = mean_squared_error(test_actuals, test_preds)
    rmse = np.sqrt(mse)
    print(f"[{input_name} Embeddings] -> Final Test MSE: {mse:.4f}  |  Test RMSE: {rmse:.4f}")
    return rmse
# 4. Calculate the Global Mean Baseline (Stupid AI)
# This guesses the *average* dataset rating (e.g., 6.4) for every single movie blindly
global_mean = train_df['vote_average'].mean()
baseline_preds = [global_mean] * len(test_df) # Make an array of thousands of guesses that are all exactly "6.4"
baseline_mse = mean_squared_error(test_df['vote_average'], baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)
print(f"\n--- Baseline Model (Blindly Guesses Global Mean: {global_mean:.2f}) ---")
print(f"Baseline Test MSE: {baseline_mse:.4f}  |  Baseline Test RMSE: {baseline_rmse:.4f}")
# Train the model on 'Overview' embeddings
rmse_overview = train_and_evaluate_regressor(train_loader_over, val_loader_over, test_loader_over, "Overview")


--- Baseline Model (Blindly Guesses Global Mean: 6.09) ---
Baseline Test MSE: 1.2825  |  Baseline Test RMSE: 1.1325

--- Training Regressor on column: 'Overview' ---
Finished Training.
[Overview Embeddings] -> Final Test MSE: 1.3408  |  Test RMSE: 1.1579


In [56]:
# 5. Repeat for at least two different single-text inputs! (We choose 'tagline')
print("\nQuickly building embeddings for the 'tagline' column...")
train_tagline_emb, val_tagline_emb, test_tagline_emb = create_document_embeddings(
    train_text=train_df['tagline_clean'],
    val_text=val_df['tagline_clean'],
    test_text=test_df['tagline_clean'],
    glove_model=glove_dict,  # Reusing our loaded Glove Model
    embed_dim=DIMENSION
)
X_train_tag_tensor = torch.tensor(train_tagline_emb, dtype=torch.float32)
X_val_tag_tensor = torch.tensor(val_tagline_emb, dtype=torch.float32)
X_test_tag_tensor = torch.tensor(test_tagline_emb, dtype=torch.float32)
train_loader_tag = DataLoader(TensorDataset(X_train_tag_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader_tag = DataLoader(TensorDataset(X_val_tag_tensor, y_val_tensor), batch_size=64)
test_loader_tag = DataLoader(TensorDataset(X_test_tag_tensor, y_test_tensor), batch_size=64)
# Train the model on 'Tagline' embeddings
rmse_tagline = train_and_evaluate_regressor(train_loader_tag, val_loader_tag, test_loader_tag, "Tagline")


Quickly building embeddings for the 'tagline' column...

--- Embedding Coverage Report ---
Dataset Unique Words (from Train set): 2994
Words Found in GloVe: 2978
Coverage: 99.47%

Calculating TF-IDF Weighted Averages...

--- Training Regressor on column: 'Tagline' ---
Finished Training.
[Tagline Embeddings] -> Final Test MSE: 1.3773  |  Test RMSE: 1.1736


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, hamming_loss
import ast

def safe_split(genre_string):
    
    if str(genre_string).startswith('['):
        try:
            return [str(g).strip() for g in ast.literal_eval(genre_string)]
        except:
            pass
    
    return [g.strip() for g in str(genre_string).split(',') if g.strip()]
print("Converting Genres to Multi-Label Checkboxes...")
train_df['genre_list'] = train_df['genres'].apply(safe_split)
val_df['genre_list'] = val_df['genres'].apply(safe_split)
test_df['genre_list'] = test_df['genres'].apply(safe_split)

mlb = MultiLabelBinarizer()

y_train_genre_numpy = mlb.fit_transform(train_df['genre_list'])
y_val_genre_numpy = mlb.transform(val_df['genre_list'])
y_test_genre_numpy = mlb.transform(test_df['genre_list'])

num_genres = len(mlb.classes_)
print(f"Total Unique Genres Found: {num_genres}")
print("Genres:", list(mlb.classes_))
# Convert to PyTorch Tensors
y_train_genre_tensor = torch.tensor(y_train_genre_numpy, dtype=torch.float32)
y_val_genre_tensor = torch.tensor(y_val_genre_numpy, dtype=torch.float32)
y_test_genre_tensor = torch.tensor(y_test_genre_numpy, dtype=torch.float32)
# Prepare DataLoaders again! (We are reusing your 'Overview' and 'Tagline' X tensors from Task 3)
train_genre_loader_over = DataLoader(TensorDataset(X_train_over_tensor, y_train_genre_tensor), batch_size=64, shuffle=True)
val_genre_loader_over = DataLoader(TensorDataset(X_val_over_tensor, y_val_genre_tensor), batch_size=64)
test_genre_loader_over = DataLoader(TensorDataset(X_test_over_tensor, y_test_genre_tensor), batch_size=64)
train_genre_loader_tag = DataLoader(TensorDataset(X_train_tag_tensor, y_train_genre_tensor), batch_size=64, shuffle=True)
val_genre_loader_tag = DataLoader(TensorDataset(X_val_tag_tensor, y_val_genre_tensor), batch_size=64)
test_genre_loader_tag = DataLoader(TensorDataset(X_test_tag_tensor, y_test_genre_tensor), batch_size=64)

Converting Genres to Multi-Label Checkboxes...
Total Unique Genres Found: 945
Genres: ['Action', 'Action Adventure', 'Action Adventure Animation Comedy Family', 'Action Adventure Animation Comedy Science Fiction', 'Action Adventure Animation Family', 'Action Adventure Animation Family Fantasy', 'Action Adventure Animation Fantasy Science Fiction', 'Action Adventure Comedy', 'Action Adventure Comedy Crime', 'Action Adventure Comedy Crime Drama', 'Action Adventure Comedy Crime Mystery', 'Action Adventure Comedy Crime Thriller', 'Action Adventure Comedy Drama Family', 'Action Adventure Comedy Drama Mystery', 'Action Adventure Comedy Drama Science Fiction', 'Action Adventure Comedy Family', 'Action Adventure Comedy Family Fantasy', 'Action Adventure Comedy Family Science Fiction', 'Action Adventure Comedy Fantasy Science Fiction', 'Action Adventure Comedy Science Fiction', 'Action Adventure Comedy Thriller', 'Action Adventure Comedy Western', 'Action Adventure Crime', 'Action Adventure Cri

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:1007: UserWarning: unknown class(es) ['Action Adventure Animation Science Fiction Thriller', 'Action Adventure Comedy Fantasy', 'Action Adventure Comedy Romance', 'Action Adventure Crime Drama', 'Action Adventure Crime Drama Family', 'Action Adventure Crime Mystery', 'Action Adventure History Romance Drama', 'Action Adventure Romance Western', 'Action Adventure Thriller War', 'Action Comedy Crime Romance', 'Action Comedy Drama Thriller', 'Action Comedy Fantasy Science Fiction', 'Action Comedy Music Romance Western', 'Action Science Fiction Comedy', 'Action Thriller Crime Mystery', 'Action Thriller Science Fiction Mystery Adventure', 'Adventure Action Comedy Thriller', 'Adventure Action Comedy Western', 'Adventure Action Horror Science Fiction Thriller', 'Adventure Action Science Fiction Thriller', 'Adventure Animation Comedy Drama', 'Adventure Animation Drama Family', 'Adventure Ani

In [ ]:
class GenreClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(GenreClassifier, self).__init__()
       
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
     
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:
def train_and_evaluate_classifier(train_loader, val_loader, test_loader, input_name):

    model = GenreClassifier(input_dim=DIMENSION, num_classes=num_genres)
    criterion = nn.BCEWithLogitsLoss() 
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    epochs = 15 # Train longer since Multi-Label is significantly harder than guessing 1 number
    print(f"\n--- Training Genre Classifier on column: '{input_name}' ---")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X) # Raw Confidences (Logits)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            
    print("Finished Training.")
    
    # 3. Final Test Set Evaluation
    model.eval()
    test_preds_binary = []
    test_actuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            raw_logits = model(batch_X)
            # Apply Sigmoid mathematically to put percentages between 0 and 1
            probabilities = torch.sigmoid(raw_logits)
            
            # Did the model cross a 50% confidence threshold? If so, True (1). Otherwise False (0).
            binary_predictions = (probabilities > 0.5).int()
            
            test_preds_binary.extend(binary_predictions.numpy())
            test_actuals.extend(batch_y.numpy())
            
    # Calculate the exact Metrics the assignment demanded
    micro_f1 = f1_score(test_actuals, test_preds_binary, average='micro', zero_division=0)
    macro_f1 = f1_score(test_actuals, test_preds_binary, average='macro', zero_division=0)
    h_loss = hamming_loss(test_actuals, test_preds_binary)
    
    print(f"[{input_name} Embeddings] -> Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f} | Hamming Loss: {h_loss:.4f}")
    return micro_f1, macro_f1, h_loss
# Run BOTH evaluations to compare!
run1 = train_and_evaluate_classifier(train_genre_loader_over, val_genre_loader_over, test_genre_loader_over, "Overview")
run2 = train_and_evaluate_classifier(train_genre_loader_tag, val_genre_loader_tag, test_genre_loader_tag, "Tagline")


--- Training Genre Classifier on column: 'Overview' ---
Finished Training.
[Overview Embeddings] -> Micro-F1: 0.0000 | Macro-F1: 0.0000 | Hamming Loss: 0.0009

--- Training Genre Classifier on column: 'Tagline' ---
Finished Training.
[Tagline Embeddings] -> Micro-F1: 0.0000 | Macro-F1: 0.0000 | Hamming Loss: 0.0009


In [67]:
from collections import Counter

# 1. Let's find the top 5 most common genres in your dataset to analyze 
all_genres_ever = []
for genres in train_df['genres'].apply(safe_split): # <-- Changed to train_df and 'genres'
    all_genres_ever.extend(genres)
    
top_5_genres = [g[0] for g in Counter(all_genres_ever).most_common(5)]
print(f"--- Analyzing the Top 5 Genres: {top_5_genres} ---\n")

def analyze_genre_frequency(target_genre):
    # Step 1: Filter to only include movies that HAVE this genre
    # We use train_df which securely has our cleaned columns!
    genre_movies = train_df[train_df['genres'].apply(lambda x: target_genre in safe_split(x))] # <-- Changed to train_df
    
    # Step 2: Dump all the "overview" text from these movies into one gigantic string
    # We know 100% that 'overview_clean' exists perfectly inside train_df
    all_text = " ".join(genre_movies['overview_clean'].tolist())
    
    # Step 3: Use Counter to count every single word's frequency
    word_counts = Counter(all_text.split())
    
    stop_words = {'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'what', 
                  'which', 'this', 'that', 'these', 'those', 'then', 'so', 'than', 'such',
                  'who', 'whom', 'whose', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
                  'have', 'has', 'had', 'do', 'does', 'did', 'to', 'from', 'in', 'out', 'on', 'off',
                  'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when',
                  'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other',
                  'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
                  'of', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
                  'before', 'after', 'above', 'below', 'up', 'down', 'by', 'at', 'it', 'they', 'them',
                  'their', 'he', 'him', 'his', 'she', 'her', 'we', 'us', 'our', 'you', 'your', 'i', 'my', 'movie', 'film', 'one'}
    
    # Filter out stop words and tiny 1-letter glitches
    clean_counts = Counter({word: count for word, count in word_counts.items() 
                           if word not in stop_words and len(word) > 2})
                           
    # Step 4: Top 10 Most Frequent 
    top_10 = clean_counts.most_common(10)
    
    # Step 5: Bottom 10 Least Frequent (With minimum threshold of 3)
    threshold_words = {word: count for word, count in clean_counts.items() if count >= 3}
    bottom_10 = sorted(threshold_words.items(), key=lambda item: item[1])[:10]
    
    # Step 6: Print out explicitly
    print(f"Genre: {target_genre.upper()} (Found {len(genre_movies)} movies)")
    print("-" * 50)
    print(f"{'Top 10 Most Frequent':<30} | {'Count':<10}")
    print("-" * 50)
    for word, count in top_10:
         print(f"{word:<30} | {count:<10}")
         
    print("\n")
    print("-" * 50)
    print(f"{'Bottom 10 (freq >= 3)':<30} | {'Count':<10}")
    print("-" * 50)
    for word, count in bottom_10:
         print(f"{word:<30} | {count:<10}")
    print("\n" + "="*50 + "\n")

# Run the function on our selected top 5 genres
for genre in top_5_genres:
    analyze_genre_frequency(genre)


--- Analyzing the Top 5 Genres: ['Drama', 'Comedy', 'Drama Romance', 'Comedy Romance', 'Comedy Drama'] ---

Genre: DRAMA (Found 257 movies)
--------------------------------------------------
Top 10 Most Frequent           | Count     
--------------------------------------------------
life                           | 106       
young                          | 49        
story                          | 47        
year                           | 47        
family                         | 43        
two                            | 34        
new                            | 34        
father                         | 32        
man                            | 32        
get                            | 31        


--------------------------------------------------
Bottom 10 (freq >= 3)          | Count     
--------------------------------------------------
farm                           | 3         
borgan                         | 3         
demand                         | 3    

## Task -6


In [69]:
from sklearn.linear_model import LogisticRegression
import numpy as np

def find_genre_indicative_words(target_genre):
    print(f"\n--- Finding Top 10 Indicative Words for: {target_genre.upper()} ---")
    
    # 1. Create a checklist for this specific genre ONLY 
    genre_labels = train_df['genres'].apply(lambda x: 1 if target_genre in safe_split(x) else 0)
    
    # 2. Train a very fast, isolated Logistic Regression model
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    # FIX: Wrap train_overview_emb in np.asarray() to convert the old matrix format correctly!
    lr.fit(np.asarray(train_overview_emb), genre_labels)
    
    # 3. We instantly recreate the tf-idf locally on training data to get exact index-to-word mapping!
    # (Because the assignment specifically asks to use raw TF-IDF to find the indicative words)
    tfidf_temp = TfidfVectorizer()
    X_temp = tfidf_temp.fit_transform(train_df['overview_clean'])
    full_vocab = {v: k for k, v in tfidf_temp.vocabulary_.items()} # Reverse lookup: {Index: "Word"}
    
    # Apply Logical Regression mapping on the raw TF-IDF to find the pure descriptive words!
    lr_raw = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_raw.fit(X_temp, genre_labels)
    raw_weights = lr_raw.coef_[0]
    
    # Find the top 10 highest positive weights mathematically
    # (argsort sorts from smallest to largest, so we look at the last 10, then reverse it)
    top_10_indexes = raw_weights.argsort()[-10:][::-1] 
    
    # Display the results
    print(f"{'Indicative Word':<20} | {'Logistic Weight Score':<10} | {'Interpretation'}")
    print("-" * 80)
    
    for idx in top_10_indexes:
        word = full_vocab[idx]
        weight = raw_weights[idx]
        
        # Give a small dynamic interpretation of why it likely matters based on magnitude
        if weight > 3.0:
            interpretation = "Extremely strong unique identifier for this specific genre!"
        elif weight > 1.5:
             interpretation = "Highly associated noun/verb for the genre."
        else:
             interpretation = "Slightly leans toward this genre."
             
        print(f"{word:<20} | +{weight:<9.2f} | {interpretation}")

# Test it magically on our top 3 genres to see the difference from Task 5!
print("Extracting Indicative Words logic using raw TF-IDF...")
for genre in top_5_genres[:3]:
    find_genre_indicative_words(genre)


Extracting Indicative Words logic using raw TF-IDF...

--- Finding Top 10 Indicative Words for: DRAMA ---
Indicative Word      | Logistic Weight Score | Interpretation
--------------------------------------------------------------------------------
life                 | +2.81      | Highly associated noun/verb for the genre.
struggle             | +2.35      | Highly associated noun/verb for the genre.
jesus                | +2.34      | Highly associated noun/verb for the genre.
player               | +2.30      | Highly associated noun/verb for the genre.
his                  | +2.13      | Highly associated noun/verb for the genre.
story                | +2.08      | Highly associated noun/verb for the genre.
drama                | +2.00      | Highly associated noun/verb for the genre.
football             | +1.97      | Highly associated noun/verb for the genre.
her                  | +1.90      | Highly associated noun/verb for the genre.
abuse                | +1.89      | High